# Iris Flower Classification Using Machine Learning

**Beginner Machine Learning Assignment**

## 1. Project idea
This project predicts the species of an Iris flower from four measurements: sepal length, sepal width, petal length, and petal width. It compares two supervised classification models: a Decision Tree and K-Nearest Neighbors (KNN).

This is useful as a beginner example of how measurements can be used to classify items. This is a demonstration dataset, not a real-world medical or safety system.

## Dataset
We use the Iris dataset bundled with scikit-learn. It contains 150 samples, four numerical features, and three species classes. Dataset documentation: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html

## 2. Import libraries

The notebook uses pandas for tables, Matplotlib and Seaborn for visualizations, and scikit-learn for the dataset, models, and evaluation metrics.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 3. Load and inspect the dataset

The four measurement columns are the **features** (`X`). The flower species is the **target** (`y`).

In [ ]:
iris = load_iris(as_frame=True)
X = iris.data.copy()
y = iris.target.copy()

df = X.copy()
df["species"] = y.map(dict(enumerate(iris.target_names)))

print("Dataset shape:", df.shape)
display(df.head())
print("\nSpecies counts:")
display(df["species"].value_counts().rename_axis("species").to_frame("count"))

## 4. Data preparation and missing-value check

We check missing values and duplicate rows. The bundled Iris dataset has no missing values. Duplicate rows are reported for transparency; they are not automatically removed because the goal is to keep the original dataset and follow a reproducible split.

In [ ]:
print("Missing values by column:")
display(df.isnull().sum().to_frame("missing_values"))

print("Number of duplicate rows:", int(df.duplicated().sum()))
print("Feature columns:", list(X.columns))
print("Target classes:", list(iris.target_names))

## 5. Split into training and testing sets

We reserve 20% of the data for testing and use `stratify=y` so each species remains proportionally represented in the train and test sets. `random_state=42` makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

## 6. Build and train two models

- **Decision Tree:** learns a sequence of feature-based decision rules.
- **KNN:** predicts based on the labels of the five nearest training samples. KNN is placed in a pipeline with `StandardScaler` so that measurements are scaled using the training data only.

In [ ]:
# Model 1: Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
dt_predictions = dt_model.predict(X_test)

# Model 2: K-Nearest Neighbors (KNN)
knn_model = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5)
)
knn_model.fit(X_train, y_train)
knn_predictions = knn_model.predict(X_test)

print("Both models have been trained.")

## 7. Evaluate and compare accuracy

Accuracy is the fraction of test examples classified correctly. The score is calculated on the held-out test set, not on the training data.

In [ ]:
dt_accuracy = accuracy_score(y_test, dt_predictions)
knn_accuracy = accuracy_score(y_test, knn_predictions)

results = pd.DataFrame({
    "Model": ["Decision Tree", "K-Nearest Neighbors (KNN)"],
    "Accuracy": [dt_accuracy, knn_accuracy],
    "Accuracy (%)": [round(dt_accuracy * 100, 2), round(knn_accuracy * 100, 2)]
})

display(results[["Model", "Accuracy (%)"]])

## 8. Confusion matrices

Rows show the actual class and columns show the predicted class. Values on the diagonal are correct predictions; values off the diagonal are mistakes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
model_predictions = [
    ("Decision Tree", dt_predictions),
    ("KNN", knn_predictions)
]

for ax, (model_name, predictions) in zip(axes, model_predictions):
    cm = confusion_matrix(y_test, predictions, labels=[0, 1, 2])
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=iris.target_names,
        yticklabels=iris.target_names,
        ax=ax
    )
    ax.set_title(f"{model_name} Confusion Matrix")
    ax.set_xlabel("Predicted species")
    ax.set_ylabel("Actual species")

plt.tight_layout()
plt.show()

## 9. Classification reports

Precision, recall, and F1-score provide additional detail beyond accuracy.

In [ ]:
print("DECISION TREE REPORT")
print(classification_report(
    y_test,
    dt_predictions,
    target_names=iris.target_names,
    zero_division=0
))

print("KNN REPORT")
print(classification_report(
    y_test,
    knn_predictions,
    target_names=iris.target_names,
    zero_division=0
))

## 10. Conclusion

The following cell identifies which model has the higher accuracy for this particular train/test split. Your conclusion should describe the result printed below. Results can vary if you change the split, random seed, or model settings. Higher accuracy on this small test set does not guarantee better performance on every future dataset.

In [ ]:
if dt_accuracy > knn_accuracy:
    winner = "Decision Tree"
    comparison = "Decision Tree achieved the higher test accuracy on this split."
elif knn_accuracy > dt_accuracy:
    winner = "K-Nearest Neighbors (KNN)"
    comparison = "KNN achieved the higher test accuracy on this split."
else:
    winner = "Tie"
    comparison = "Both models achieved the same test accuracy on this split."

print(f"Decision Tree accuracy: {dt_accuracy * 100:.2f}%")
print(f"KNN accuracy: {knn_accuracy * 100:.2f}%")
print("Best result on this split:", winner)
print(comparison)
print(
    "\nWhat I learned: I learned how to load a dataset, inspect missing values, "
    "separate features and labels, split data into training and testing sets, "
    "train two classification models, and compare their performance with accuracy "
    "and confusion matrices."
)

## References
- scikit-learn Iris dataset documentation: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html
- scikit-learn model evaluation documentation: https://scikit-learn.org/stable/modules/model_evaluation.html